Análise Inicial

In [ ]:
import pandas as pd
import numpy as np
import datetime as dt
import plotly.express as px

In [ ]:
df = pd.read_excel('../data/raw/amazon_delivery_dados_alunos.xlsx')

In [ ]:
df = pd.read_csv('../data/raw/amazon_delivery.csv')

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.isnull().sum()

In [ ]:
df.value_counts()

In [ ]:
df.describe(include='object')

___

Análise Exploratória

In [ ]:
df = pd.read_parquet('../data/processed/amazon_delivery.parquet')

In [ ]:
df.info()

In [ ]:
df['Order_ID'].nunique()

In [ ]:
round(df['Delivery_Time'].min().item(), 0)

In [ ]:
round(df['Delivery_Time'].max().item(), 0)

In [ ]:
round(df['Delivery_Time'].std().item(), 0)

In [ ]:
round(df['Delivery_Time'].mean().item(), 0)

In [ ]:
# Qual o tempo médio e as variações das entregas?

round(df['Delivery_Time'].describe(),0)

In [ ]:
# Qual a quantidade de entregas com e sem atraso?
df_grafico = df.groupby(['Order_Week', 'Delivery_Status'])['Order_ID'].nunique().reset_index()

# 3. Mapear as cores exatas da imagem
cores = {'ontime': '#4A90E2', 'delay': '#F5A623'}

# 4. Criar o gráfico de área
fig = px.line(
    df_grafico,
    x='Order_Week', 
    y='Order_ID', 
    color='Delivery_Status',
    color_discrete_map=cores,
    title='Entrega por Status',
    category_orders={
        "Delivery_Status": ["on_time", "delay"]
    },
    labels={
        'Order_Week': 'Semanas',
        'Order_ID': 'Quantidade de Pedidos'
    },
)

# 5. Adicionar os marcadores e rótulos de dados (os números sobre as linhas)
fig.update_traces(
    mode="markers+lines+text", 
    texttemplate='%{y}', 
    textposition="top center",
)

# 6. Ajustes estéticos de layout
fig.update_layout(
    hovermode='x unified',
    plot_bgcolor='white',
    xaxis=dict(
        type='category'
    ),
    yaxis=dict(
        gridcolor='lightgrey',
        showgrid=True,
        # range=[0, gasto_etario['Gasto-Cliente'].max() * 1.2],
        zeroline=True,
        zerolinecolor='black',
    )
)

fig.show()

In [ ]:
# Em quais região/área o atraso se concentra?

df_area = df.groupby(['Area', 'Delivery_Status'])['Order_ID'].nunique().reset_index(name='Quantidade')

cores = {'ontime': '#4A90E2', 'delay': '#F5A623'}

# 4. Criar o gráfico de área
df_area['Total_Area'] = df_area.groupby('Area')['Quantidade'].transform('sum')
# Calculamos quanto o status representa do total daquela área
df_area['Percentual'] = (df_area['Quantidade'] / df_area['Total_Area']) * 100

areas = [
    'Metropolitian', 'Urban', 'Other', 'Semi-Urban'
]

# 3. Criar o gráfico usando a coluna 'Percentual' no eixo Y
fig = px.bar(
    df_area, 
    x='Area', 
    y='Percentual', 
    color='Delivery_Status',
    title='% Atraso por Área',
    text=df_area['Quantidade'],
    color_discrete_map={
        'ontime': '#4A90E2',
        'delay': '#F5A623'
    },
    category_orders={
        "Area": areas,
        "Delivery_Status": ["ontime", "delay"] # Altera qual cor fica embaixo/em cima
    }
)

fig.update_traces(
    textposition='inside',      # Força o texto para dentro da barra
    textangle=0,
    # insidetextanchor='center',  # Centraliza o texto na fatia
    textfont_size=12,           # Define um tamanho fixo
    textfont_color='white',     # Cor branca para contraste (ou black se preferir)
    cliponaxis=False            # Impede que o texto seja cortado nas bordas
)

fig.for_each_trace(lambda t: t.update(
    textfont=dict(color="white", size=12) if t.name == "ontime" 
    else dict(color="black", size=12)
))

# 4. Ajustes para "extender" e colar as barras
fig.update_layout(
    barmode='stack', # Empilha as barras para somarem 100%
    bargap=0.1,      # Deixa as barras mais largas
    plot_bgcolor='white',
    yaxis=dict(
        ticksuffix="%",
        range=[0, df_area['Percentual'].max() * 1.2],
        showgrid=True,
        gridcolor='black',
        gridwidth=1,
        zeroline=True,             # <-- mostra a linha do eixo 0
        zerolinecolor='black',     # <-- define a cor
        zerolinewidth=1.5,
    ) # Garante que o eixo Y vá até 100
)

fig.show()

In [ ]:
# Em quais clima o atraso se concentra?
df_clima = df.groupby(['Weather', 'Delivery_Status'])['Order_ID'].nunique().reset_index(name='Quantidade')

df_clima['Total_Area'] = df_clima.groupby('Weather')['Quantidade'].transform('sum')
# Calculamos quanto o status representa do total daquela área
df_clima['Percentual'] = (df_clima['Quantidade'] / df_clima['Total_Area']) * 100

clima = [
    'Fog', 'Stormy','Cloudy', 'Sandstorms', 'Windy', 'Sunny', 'NaN'
]

# cores_texto = ['white' if status == 'ontime' else 'black' for status in df_clima['Delivery_Status']]

fig = px.bar(
    df_clima, 
    x='Percentual', 
    y='Weather', 
    color='Delivery_Status',
    title='% Atraso por Clima',
    text=df_clima['Quantidade'], # Rótulo com %
    color_discrete_map={
        'ontime': '#4A90E2',
        'delay': '#F5A623'
    },
    # orientation='h'
    category_orders={
        "Weather": clima,
        "Delivery_Status": ["ontime", "delay"] # Altera qual cor fica embaixo/em cima
    }
)

fig.for_each_trace(lambda t: t.update(textfont_color="white") if t.name == "ontime" else t.update(textfont_color="black"))

fig.update_xaxes(
    tickformat=".0f",        # Remove casas decimais dos números do eixo
    ticksuffix="%",          # Adiciona o símbolo de %
    range=[0, 101.1],          # Garante que o eixo termine em 100%,
    # tick0=0,
    dtick=10,
    # tickvals=[0, 20, 40, 60, 80, 100],
    showline=True,           # Mostra a linha do eixo
    linewidth=1, 
    linecolor='lightgrey', 
    showgrid=True,           # Ativa as linhas de grade verticais
    gridcolor='Black',
    zeroline=True,           # Ativa a linha do "zero"
    zerolinewidth=2, 
    zerolinecolor='grey',     # Cor da linha vertical inicial (Y-axis line)
    title_text=""
)

fig.update_yaxes(
    showline=True, 
    linewidth=1, 
    linecolor='Black',
    title_text=""            # Remove o título "Weather" para limpar o visual
)

fig.update_traces(
    textposition='inside',
    texttemplate='%{text}',  # Garante que use o valor passado em 'text' (Quantidade)
    insidetextanchor='end',  # Alinha o texto à direita dentro da barra (opcional)
)

fig.show()

In [ ]:
# Em quais tráfego o atraso se concentra?
df_trafego = df.groupby(['Traffic', 'Delivery_Status'])['Order_ID'].nunique().reset_index(name='Quantidade')

df_trafego['Total_Area'] = df_trafego.groupby('Traffic')['Quantidade'].transform('sum')
# Calculamos quanto o status representa do total daquela área
df_trafego['Percentual'] = (df_trafego['Quantidade'] / df_trafego['Total_Area']) * 100

trafego = [
    'Low', 'Jam', 'Medium', 'High'
]

fig = px.bar(
    df_trafego, 
    x='Percentual', 
    y='Traffic', 
    color='Delivery_Status',
    title='% Atraso por Tráfego',
    text=df_trafego['Quantidade'], # Rótulo com %
    color_discrete_map={'ontime': '#4A90E2', 'delay': '#F5A623'},
    # orientation='h'
    category_orders={
        "Traffic": trafego,
        "Delivery_Status": ["ontime", "delay"] # Altera qual cor fica embaixo/em cima
    }
)

fig.for_each_trace(lambda t: t.update(textfont_color="white") if t.name == "ontime" else t.update(textfont_color="black"))

fig.update_xaxes(
    tickformat=".0f",        # Remove casas decimais dos números do eixo
    ticksuffix="%",          # Adiciona o símbolo de %
    range=[0, 101.1],          # Garante que o eixo termine em 100%,
    # tick0=0,
    dtick=10,
    # tickvals=[0, 20, 40, 60, 80, 100],
    showline=True,           # Mostra a linha do eixo
    linewidth=1, 
    linecolor='lightgrey', 
    showgrid=True,           # Ativa as linhas de grade verticais
    gridcolor='Black',
    zeroline=True,           # Ativa a linha do "zero"
    zerolinewidth=2, 
    zerolinecolor='grey',     # Cor da linha vertical inicial (Y-axis line)
    title_text=""
)

fig.update_yaxes(
    showline=True, 
    linewidth=1, 
    linecolor='Black',
    title_text=""            # Remove o título "Weather" para limpar o visual
)

fig.update_traces(
    textposition='inside',
    texttemplate='%{text}',  # Garante que use o valor passado em 'text' (Quantidade)
    insidetextanchor='end',  # Alinha o texto à direita dentro da barra (opcional)
)


fig.show()

In [ ]:
# Em quais veículo o atraso se concentra?
df_veiculo = df.groupby(['Vehicle', 'Delivery_Status'])['Order_ID'].nunique().reset_index(name='Quantidade')

df_veiculo['Total_Area'] = df_veiculo.groupby('Vehicle')['Quantidade'].transform('sum')
# Calculamos quanto o status representa do total daquela área
df_veiculo['Percentual'] = (df_veiculo['Quantidade'] / df_veiculo['Total_Area']) * 100

veiculo = [
    'motorcycle', 'scooter', 'van', 'bicycle'
]

fig = px.bar(
    df_veiculo, 
    x='Percentual', 
    y='Vehicle', 
    color='Delivery_Status',
    title='% Atraso por Veículo',
    text=df_veiculo['Quantidade'], # Rótulo com %
    color_discrete_map={'ontime': '#4A90E2', 'delay': '#F5A623'},
    # orientation='h'
    category_orders={
        "Vehicle": veiculo,
        "Delivery_Status": ["ontime", "delay"] # Altera qual cor fica embaixo/em cima
    }
)
fig.for_each_trace(lambda t: t.update(textfont_color="white") if t.name == "ontime" else t.update(textfont_color="black"))

fig.update_xaxes(
    tickformat=".0f",        # Remove casas decimais dos números do eixo
    ticksuffix="%",          # Adiciona o símbolo de %
    range=[0, 101.1],          # Garante que o eixo termine em 100%,
    # tick0=0,
    dtick=10,
    # tickvals=[0, 20, 40, 60, 80, 100],
    showline=True,           # Mostra a linha do eixo
    linewidth=1, 
    linecolor='lightgrey', 
    showgrid=True,           # Ativa as linhas de grade verticais
    gridcolor='Black',
    zeroline=True,           # Ativa a linha do "zero"
    zerolinewidth=2, 
    zerolinecolor='grey',     # Cor da linha vertical inicial (Y-axis line)
    title_text=""
)

fig.update_yaxes(
    showline=True, 
    linewidth=1, 
    linecolor='Black',
    title_text=""            # Remove o título "Weather" para limpar o visual
)

fig.update_traces(
    textposition='inside',
    texttemplate='%{text}',  # Garante que use o valor passado em 'text' (Quantidade)
    insidetextanchor='end',  # Alinha o texto à direita dentro da barra (opcional)
)


fig.show()

In [ ]:
# Tabela dinâmica de Status de Entrega por Clima
pivot_clima = df.pivot_table(index=['Area', 'Weather', 'Vehicle'], 
                             columns='Delivery_Status', 
                             values='Order_ID',
                             aggfunc='count',
                             fill_value=0
                             )

# Aplicar o mapa de calor com cores verdes (mais alto = melhor)
pivot_clima_estilo = pivot_clima.style \
    .set_caption("Status de Entrega por Clima") \
    .background_gradient(cmap='RdYlGn', axis=None) \
    .format("{:.2f}") # Opcional: formata para 2 casas decimais

pivot_clima_estilo

In [ ]:
# Existe relação com idade média do entregador?
# Agrupando os dados para obter as médias e a contagem (tamanho da bolha)
tabela_bolhas = df.groupby('Delivery_Status').agg({
    'Agent_Age': 'mean',
    'Delivery_Time': 'mean',
    'Order_ID': 'count'
}).reset_index()

# Renomeando para facilitar a leitura no gráfico
tabela_bolhas.columns = ['Status', 'Idade_Media', 'Tempo_Medio', 'Total_Pedidos']

fig = px.scatter(
    tabela_bolhas,
    x='Idade_Media',
    y='Tempo_Medio',
    size='Total_Pedidos',      # O tamanho da bolha depende do volume de pedidos
    color='Status',            # Cor baseada no status
    text='Status',             # Texto dentro/sobre a bolha
    labels={
        'Idade_Media': 'Agent_Age',
        'Tempo_Medio': 'Delivery_Time'
    },
    range_x=[20, 38],          # Ajuste conforme a imagem (20 a 38)
    range_y=[25, 275],         # Ajuste conforme a imagem (até 250+)
    title="Atraso Médio por Idade"
)

# Ajustes estéticos para ficar idêntico à imagem
fig.update_traces(
    textposition='top center', 
    marker=dict(sizeref=2.*max(tabela_bolhas['Total_Pedidos'])/(60**2), opacity=0.6)
)

# Deixar o fundo branco com linhas de grade (estilo Looker/Google)
fig.update_layout(
    plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridcolor='lightgrey'),
    yaxis=dict(
        showgrid=True,
        gridcolor='lightgrey',
        gridwidth=1,
        zeroline=True,             # <-- mostra a linha do eixo 0
        zerolinecolor='black',     # <-- define a cor
        zerolinewidth=1.5,
    ),
    showlegend=True
)

fig.show()

In [ ]:
# Como é a distribuição das entregas por categoria?
# Em quais categoria o atraso se concentra?
categorias_manter = [
    'Electronics', 'Books', 'Jewelry', 'Toys', 'Skincare', 
    'Snacks', 'Outdoors', 'Apparel', 'Sports'
]

# Criar uma cópia da coluna para não estragar os dados originais
df['Category_Grouped'] = df['Category'].apply(lambda x: x if x in categorias_manter else 'Outros')

# 2. Gerar a tabela para o gráfico usando a nova coluna
tabela_categoria = df.groupby('Category_Grouped')['Order_ID'].count().reset_index()
tabela_categoria.columns = ['Categoria', 'Qtd Entregas']

tabela_categoria['ordem_aux'] = tabela_categoria['Categoria'].apply(lambda x: 0 if x == 'Outros' else 1)
tabela_categoria = tabela_categoria.sort_values(by=['ordem_aux', 'Qtd Entregas'], ascending=[True, False])

fig = px.pie(
    tabela_categoria,
    names='Categoria',
    values='Qtd Entregas',
    title='Entregas por Categoria',
    color_discrete_sequence=px.colors.qualitative.Pastel,
    hole=.5
)

fig.update_traces(
    rotation=200, 
    direction='clockwise', 
    textinfo='percent'
)

fig.show()

In [ ]:
# Quais as avaliações médias das entregas com e sem atraso nos segmento em que acontece atraso?
pivot_rating = df.pivot_table(index=['Area', 'Vehicle'], 
                             columns='Delivery_Status', 
                             values='Agent_Rating',
                             aggfunc='mean',
                             fill_value=0
                             )

# Aplicar o mapa de calor com cores verdes (mais alto = melhor)
pivot_rating_estilo = pivot_rating.style \
    .set_caption("Avaliações médias das entregas com e sem atraso") \
    .background_gradient(cmap='RdYlGn', axis=None) \
    .format("{:.2f}") # Opcional: formata para 2 casas decimais

pivot_rating_estilo

___

In [ ]:
'''
# Exercício 1 — Evolução dos atrasos ao longo do tempo

Tare
Faça um gráfico de barras verticais usando as colunas Order_Week, Order_ID e Delivery_Status.

- Considere apenas entregas com status delay.
- Agrupe os dados por semana (Order_Week).
- Conte a quantidade de pedidos atrasados (Order_ID).

Pergunta de negócio
Os atrasos estão aumentando, diminuindo ou se mantendo estáveis ao longo das semanas?

'''

df_atrasos = df[df['Delivery_Status'] == 'delay'].groupby('Order_Week')['Order_ID'].nunique().reset_index(name='Quantidade')

fig = px.bar(
    df_atrasos,
    x='Order_Week',
    y='Quantidade',
    text='Quantidade',
    title='Os atrasos estão aumentando, diminuindo ou se mantendo estáveis ao longo das semanas?',
)

fig.update_xaxes(
    type='category',
)

fig.show()

In [ ]:
'''
Exercício 2 — Comparação entre entregas no prazo e atrasadas

Tarefa
Compare a quantidade de entregas no prazo e atrasadas por semana.
Faça um gráfico de barras empilhadas verticais usando as colunas Order_Week, Order_ID 
e Delivery_Status.

- Agrupe por semana (Order_Week).
- Separe os dados por status de entrega (ontime e delay).

Pergunta de negócio
Em quais semanas a proporção de atrasos foi mais crítica?
'''

df_entregas = df.groupby(['Order_Week', 'Delivery_Status'])['Order_ID'].nunique().reset_index(name='Quantidade')

fig = px.bar(
    df_entregas,
    x='Order_Week',
    y='Quantidade',
    text='Quantidade',
    title='Em quais semanas a proporção de atrasos foi mais crítica?',
    color='Delivery_Status',
    color_discrete_map={'ontime': '#4A90E2', 'delay': '#F5A623'}
)

fig.update_xaxes(
    type='category', # Força a interpretação como texto/categoria e não data
)

fig.show()

In [ ]:
'''
Exercício 3 — Impacto do clima nos atrasos

Tarefa
Analise como as condições climáticas influenciam os atrasos nas entregas.
Faça um gráfico de barras horizontais mostrando o percentual de atrasos por condição climática
usando as colunas Weather, Order_ID e Delivery_Status.

- Para cada condição climática, calcule o percentual de pedidos com status delay.
- Ordene do maior para o menor percentual de atraso.

Pergunta de negócio
Quais condições climáticas representam maior risco de atraso para a operação?
'''

df_risco = df.groupby(['Weather', 'Delivery_Status'])['Order_ID'].nunique().reset_index(name='Qtd')
df_risco['Total'] = df_risco.groupby('Weather')['Qtd'].transform('sum')
df_risco['Taxa_Atraso'] = (df_risco['Qtd'] / df_risco['Total']) * 100

df_atraso = df_risco[df_risco['Delivery_Status'] == 'delay'].sort_values('Taxa_Atraso', ascending=True)

# Ordenar para que os climas com MAIOR taxa de atraso apareçam primeiro
# ordem_risco = df_risco[df_risco['Delivery_Status'] == 'delay'].sort_values('Taxa_Atraso', ascending=False)['Weather'].tolist()

# 2. Criação do Gráfico
fig = px.bar(
    df_atraso, 
    y='Weather', 
    x='Taxa_Atraso', 
    color='Delivery_Status',
    orientation='h',
    title='<b>Quais condições climáticas representam maior risco de atraso para a operação?</b>',
    text=df_atraso['Taxa_Atraso'].apply(lambda x: f'{x:.1f}%'),
    # category_orders={"Weather": ordem_risco, "Delivery_Status": ["ontime", "delay"]},
    color_discrete_map={'ontime': '#4A90E2', 'delay': '#F5A623'}
)

# 3. Estilização para Resposta Executiva
fig.update_layout(
    font=dict(family="Arial", size=14),
    plot_bgcolor='white',
    bargap=0.2,
    xaxis=dict(title="Probabilidade de Atraso", ticksuffix="%", range=[0, 105], dtick=20, showgrid=True, gridcolor='lightgrey'),
    yaxis=dict(title=None),
    legend_title_text='Status'
)

# Forçar cores de texto uniformes (Branco no azul, Preto no laranja)
fig.for_each_trace(lambda t: t.update(textfont=dict(color="white") if t.name == "ontime" else dict(color="black")))

fig.show()

In [ ]:
'''
Exercício 4 — Veículos e eficiência de entrega

Tarefa
Avalie a eficiência dos diferentes tipos de veículos utilizados nas entregas.
Faça um gráfico de barras empilhadas horizontais usando as colunas Vehicle, Order_ID 
e Delivery_Status.

Orientação
- Mostre a proporção de entregas ontime e delay para cada tipo de veículo.
- Use barras normalizadas em 100%.

Pergunta de negócio
Qual tipo de veículo apresenta a maior taxa de atrasos?
'''

# 1. Agrupamento e Cálculo de Proporção
df_veiculo = df.groupby(['Vehicle', 'Delivery_Status'])['Order_ID'].nunique().reset_index(name='Quantidade')

# Criamos a base para a normalização (100%)
df_veiculo['Total_Veiculo'] = df_veiculo.groupby('Vehicle')['Quantidade'].transform('sum')
df_veiculo['Percentual'] = (df_veiculo['Quantidade'] / df_veiculo['Total_Veiculo']) * 100

# 2. Criar o gráfico de barras empilhadas 100%
fig = px.bar(
    df_veiculo, 
    x='Percentual', 
    y='Vehicle', 
    color='Delivery_Status',
    orientation='h',
    title='<b>Qual tipo de veículo apresenta a maior taxa de atrasos?</b>',
    text=df_veiculo['Percentual'].apply(lambda x: f'{x:.1f}%'),
    color_discrete_map={
        'ontime': '#4A90E2',
        'delay': '#F5A623'
    },
    category_orders={
        "Delivery_Status": ["ontime", "delay"]
    }
)

# 3. Aplicar a padronização de fonte e cores de texto
fig.for_each_trace(lambda t: t.update(
    textfont=dict(color="white") if t.name == "ontime" else dict(color="black")
))

# 4. Ajustes de Layout e Eixos
fig.update_layout(
    font=dict(family="Arial", size=14, color="black"),
    plot_bgcolor='white',
    barmode='stack',
    bargap=0.2,
    xaxis=dict(
        title="Proporção da Operação (%)",
        ticksuffix="%",
        range=[0, 100.5],
        dtick=20,
        showgrid=True,
        gridcolor='lightgrey',
        zeroline=True,
        zerolinecolor='black',
        zerolinewidth=2
    ),
    yaxis=dict(title=None),
    legend_title_text='Status',
    title_x=0.5,
    uniformtext_minsize=12,
    uniformtext_mode='hide'
)

fig.update_traces(
    textposition='inside',
    cliponaxis=False
)

fig.show()

In [ ]:
'''
Exercício 5 — Atrasos por área de entrega

Tarefa
Identifique quais áreas possuem maior concentração de entregas atrasadas.
Faça um gráfico de barras verticais empilhadas usando as colunas Area, Order_ID e Delivery_Status.

Orientação
- Calcule a quantidade de entregas por área.
- Separe os resultados por status (ontime e delay).

Pergunta de negócio
Em quais áreas a empresa deveria priorizar ações para reduzir atrasos?
'''

# 1. Agrupamento para identificar concentração de volume por área e status
df_area_abs = df.groupby(['Area', 'Delivery_Status'])['Order_ID'].nunique().reset_index(name='Quantidade')

# 2. Ordenação para destacar as áreas com maior volume total (prioridade logística)
df_area_abs['Total_Area'] = df_area_abs.groupby('Area')['Quantidade'].transform('sum')
df_area_abs = df_area_abs.sort_values(by='Total_Area', ascending=False)

# 3. Criar o gráfico de barras verticais empilhadas
fig = px.bar(
    df_area_abs, 
    x='Area', 
    y='Quantidade', 
    color='Delivery_Status',
    title='<b>Em quais áreas a empresa deveria priorizar ações para reduzir atrasos?</b>',
    text='Quantidade',
    color_discrete_map={'ontime': '#4A90E2', 'delay': '#F5A623'},
    category_orders={"Delivery_Status": ["ontime", "delay"]},
)

# 4. Ajustes de Estilo e Fonte Uniforme
fig.update_layout(
    font=dict(family="Arial", size=14, color="black"),
    plot_bgcolor='white',
    barmode='group',
    bargap=0.3,
    title_x=0.5,
    yaxis=dict(
        title="Quantidade de Pedidos",
        showgrid=True,
        gridcolor='lightgrey',
        zeroline=True,
        zerolinecolor='black',
        range=[0, df_area['Quantidade'].max() * 1.4]
    ),
    xaxis=dict(title=None),
    uniformtext_minsize=10,
    uniformtext_mode='show',
)

fig.update_traces(
    textposition='outside',
    texttemplate='%{text}',
    cliponaxis=False
)

fig.show()

___